In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 550.1/550.1 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install networkx

In [ ]:
from pathlib import Path
import os
# from dotenv import load_dotenv
# load_dotenv()

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
DATA_PATH = Path("/content/drive/MyDrive/Colab Notebooks/data/council.txt")
text = DATA_PATH.read_text(encoding="utf-8")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib import font_manager
# 한글 폰트 (시스템에 따라 다름)
for f in ["NanumGothic", "Noto Sans CJK KR", "AppleGothic", "Malgun Gothic"]:
    if any(f in fn.name for fn in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = f; break
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
G_demo = nx.DiGraph() # Di는 Directional

G_demo.add_node('모리카이 센', type = 'Person')
G_demo.add_node('콰빅스 연구소', type = 'Organization')
G_demo.add_node('브렌술 아카테미', type = 'Organization')
G_demo.add_node('탈라미르 헤쉬', type = 'Person')
G_demo.add_node('벨트란 사고 엔진', type = 'Product')

G_demo.add_edge('모리카이 센', '콰빅스 연구소', relation = 'founded', year = 1987)
G_demo.add_edge('모리카이 센', '탈라미르 헤쉬', relation = 'mentor_of')
G_demo.add_edge('모리카이 센', '브렌술 아카테미', relation = 'postdoc_at')
G_demo.add_edge('탈라미르 헤쉬', '브렌술 아카테미', relation = 'founded')
G_demo.add_edge('콰빅스 연구소', '벨트란 사고 엔진', relation = 'developed')

## Page Rank 알고리즘
- 구글 검색결과 관련 알고리즘

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50, separators = ['\n\n', '\n', '. ', ' ',''])
chunks =  splitter.split_text(text)

In [ ]:
prompt_skel = """You are an information extraction system.
From the text, extract entities and relations as JSON.
IMPORTANT:
- Always extract titles in quotation marks (papers, books, protocols, projects, products)
  as Concept/Publication entities — these are easy to miss.
- When "Person X wrote/authored Title Y" appears, emit an explicit
  (X) --[wrote]--> (Y) relation.
Schema:
{
  "entities": [{"name": "...", "type": "Person|Organisation|Product|Concept|Publication"}],
  "relations": [{"source": "...", "target": "...", "label": "..."}]
}
Output ONLY valid JSON. No other text.
Text:
\"\"\"모리카이 센은 1987년에 콰빅스 연구소를 설립했다.\"\"\""""

In [ ]:
import re
import json

In [ ]:
def strip_code_fence(text):
  text = text.strip()
  if text.startswith("```"):
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
  return text


def extract_graph(text):
    system_msg = SystemMessage(content="""
You are an information extraction system.
From the text, extract entities and relations as JSON.

IMPORTANT:
- Always extract titles in quotation marks (papers, books, protocols, projects, products)
  as Concept/Publication entities — these are easy to miss.
- When "Person X wrote/authored Title Y" appears, emit an explicit
  (X) --[wrote]--> (Y) relation.
""")

    user_msg = HumanMessage(content=f"""
Schema:
{{
  "entities": [
    {{"name": "...", "type": "Person|Organisation|Product|Concept|Publication"}}
  ],
  "relations": [
    {{"source": "...", "target": "...", "label": "..."}}
  ]
}}

Output ONLY valid JSON. No other text.

Text:
\"\"\"{text}\"\"\"
""")

    raw = llm.invoke([system_msg, user_msg]).content

    try:
        return json.loads(strip_code_fence(raw))
    except json.JSONDecodeError:
        return {
            "entities": [],
            "relations": []
        }

In [ ]:
para1 = text.split("제목")[1]

In [ ]:
para1

': 콰빅스 연구소\n모리카이 센은 1987 년 초여름에 눌마리 섬에서 콰빅스 연구소를 설립했다. 센은 콰빅스를 세우기 전 11 년 동안 브렌술 아카데미의 박사후 연구원으로 지냈으며, 그 시절 그녀의 지도 멘토는 탈라미르 헤쉬였다. 헤쉬는 센이 발표한 초기 논문 두 편의 지도 교수였고, 공식 기록에도 센의 박사 학위 지도 교수로 등재되어 있다. 콰빅스의 대표 프로젝트는 기호 추론 시스템인 벨트란 사고 엔진이며, 현재 눌마리 연구 위원회가 주요 후원 기관 중 하나로 올라 있다.\n\n'

In [ ]:
result = extract_graph("제목" + para1)

In [ ]:
print(f"entities : {len(result['entities'])}, relations : {len(result['relations'])}")

entities : 5, relations : 5


In [ ]:
print(json.dumps(result, indent = 2, ensure_ascii = False))

{
  "entities": [
    {
      "name": "모리카이 센",
      "type": "Person"
    },
    {
      "name": "탈라미르 헤쉬",
      "type": "Person"
    },
    {
      "name": "콰빅스 연구소",
      "type": "Organisation"
    },
    {
      "name": "벨트란 사고 엔진",
      "type": "Product"
    },
    {
      "name": "브렌술 아카데미",
      "type": "Organisation"
    }
  ],
  "relations": [
    {
      "source": "모리카이 센",
      "target": "콰빅스 연구소",
      "label": "founded"
    },
    {
      "source": "모리카이 센",
      "target": "브렌술 아카데미",
      "label": "worked at"
    },
    {
      "source": "탈라미르 헤쉬",
      "target": "모리카이 센",
      "label": "mentor"
    },
    {
      "source": "탈라미르 헤쉬",
      "target": "모리카이 센",
      "label": "supervisor"
    },
    {
      "source": "모리카이 센",
      "target": "벨트란 사고 엔진",
      "label": "project of"
    }
  ]
}


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal # 선택지 강제

In [ ]:
# 데이터 스키마
class Entity(BaseModel):
  name : str
  type : Literal["Person", "Organization", "Product", "Concept", "Publication"]

class Relation(BaseModel):
  source : str
  target : str
  lavel : str

class ExtractedKG(BaseModel):
  entities : list[Entity] # list of Entity
  relations : list[Relation] # list of Relation

In [ ]:
structured_llm = llm.with_structured_output(ExtractedKG)

In [ ]:
para1

': 콰빅스 연구소\n모리카이 센은 1987 년 초여름에 눌마리 섬에서 콰빅스 연구소를 설립했다. 센은 콰빅스를 세우기 전 11 년 동안 브렌술 아카데미의 박사후 연구원으로 지냈으며, 그 시절 그녀의 지도 멘토는 탈라미르 헤쉬였다. 헤쉬는 센이 발표한 초기 논문 두 편의 지도 교수였고, 공식 기록에도 센의 박사 학위 지도 교수로 등재되어 있다. 콰빅스의 대표 프로젝트는 기호 추론 시스템인 벨트란 사고 엔진이며, 현재 눌마리 연구 위원회가 주요 후원 기관 중 하나로 올라 있다.\n\n'

In [ ]:
demo_kg = structured_llm.invoke(f"다음 텍스트에서 엔티티/관계를 추출하세요:\n{para1}")

In [ ]:
demo_kg.entities

[Entity(name='콰빅스 연구소', type='Organization'),
 Entity(name='모리카이 센', type='Person'),
 Entity(name='브렌술 아카데미', type='Organization'),
 Entity(name='탈라미르 헤쉬', type='Person'),
 Entity(name='벨트란 사고 엔진', type='Product'),
 Entity(name='눌마리 연구 위원회', type='Organization')]

In [ ]:
# MS GraphRAG 참고

In [ ]:
demo_kg.relations

[Relation(source='모리카이 센', target='콰빅스 연구소', lavel='설립'),
 Relation(source='모리카이 센', target='브렌술 아카데미', lavel='박사후 연구원'),
 Relation(source='탈라미르 헤쉬', target='모리카이 센', lavel='멘토'),
 Relation(source='탈라미르 헤쉬', target='모리카이 센', lavel='지도 교수'),
 Relation(source='콰빅스 연구소', target='벨트란 사고 엔진', lavel='대표 프로젝트'),
 Relation(source='눌마리 연구 위원회', target='콰빅스 연구소', lavel='후원 기관')]

In [ ]:
def extracted_graph_typed(text) -> ExtractedKG:
  # 엔티티 데이터 스키마를 이용해서 데이터 스키마에 맞는 엔티티와 릴레이션을 뽑는/..?
  #

In [ ]:
def extracted_graph_typed(text) -> ExtractedKG:
  system_msg = SystemMessage(content="""
  You are an information extraction system.
  From the text, extract entities and relations.

  IMPORTANT:
  - Always extract titles in quotation marks (papers, books, protocols, projects, products)
    as Concept/Publication entities — these are easy to miss.
  - When "Person X wrote/authored Title Y" appears, emit an explicit
    (X) --[wrote]--> (Y) relation.
  """)
  user_msg = HumanMessage(content=f'다음 텍스트에서 엔티티/관계를 추출하세요:\n"""{text}"""')

  structured_llm = llm.with_structured_output(ExtractedKG)
  result = structured_llm.invoke([system_msg, user_msg])
  return result

In [ ]:
ans = extracted_graph_typed(para1)

In [ ]:
ans.entities

[Entity(name='콰빅스 연구소', type='Organization'),
 Entity(name='모리카이 센', type='Person'),
 Entity(name='브렌술 아카데미', type='Organization'),
 Entity(name='탈라미르 헤쉬', type='Person'),
 Entity(name='벨트란 사고 엔진', type='Product'),
 Entity(name='눌마리 연구 위원회', type='Organization')]

In [ ]:
mini_dict = {
     'entities' : [{'name': 'A', 'type' : 'Person'}, {'name' : 'B', 'type' : 'Organisation'}],
     'relations' : [{'source': 'A', 'target' : 'B', 'label' : 'founded'}]
 }

In [ ]:
G_mini = nx.DiGraph()

for e in mini_dict['entities']:
  G_mini.add_node(e['name'], type = e['type'])

for r in mini_dict['relations']:
  G_mini.add_edge(r['source'], r['target'], relation = r['label'])

In [ ]:
dict(G_mini.nodes(data=True))

{'A': {'type': 'Person'}, 'B': {'type': 'Organisation'}}

In [ ]:
def json_to_graph(data) -> nx.DiGraph:
  # json을 그래프로 생성? 일반화하는 함수

In [ ]:
def json_to_graph(data) -> nx.DiGraph:
  G = nx.DiGraph()

  for e in data.get('entities'):
    G.add_node(e['name'], type=e.get('type'))

  for r in data('relations'):
    # 일반화된 함수를 작성할 때는 예외 처리를 잘 할것.
    if r['source'] not in G:
      G.add_node(r['source'], type = 'unknown')
    if r['target'] not in G:
      G.add_node(r['target'], type = 'unknown')

    G.add_edge(r['source'], r['target'], relation=r.get('label'))

  return G



In [ ]:
json_d = json.dumps(result, indent = 2, ensure_ascii = False)

In [ ]:
data = json.loads(json_d)   # 문자열 → dict
G = json_to_graph(data)

TypeError: 'dict' object is not callable

In [ ]:
# (a, b, c)

In [ ]:
# 특정 단어, 동의어, 객체 등등
# 모리카이 센 -> 뜻/객체는 동일하지만 다양한 표현이 있을 수 있음
  # 모리카이
  # 센
  # 모리카이센

In [ ]:
nodes = ['모리카이 센', '센', '콰빅스 연구소', '콰빅스', '벨트란 사고 엔진']


In [ ]:
def find_aliases(nodes):
  aliases = {}
  for short in nodes:
    for long in nodes:
      if short != long and len(short) < len(long) and short in long:
        aliases[short] = long # aliases['센'] = '모리카이 센'
        break

  return aliases

In [ ]:
find_aliases(nodes)

{'센': '모리카이 센', '콰빅스': '콰빅스 연구소'}

In [ ]:
def resolve_entities(G: nx.DiGraph) -> nx.DiGraph:
    nodes = list(G.nodes)
    aliases = find_aliases(nodes)

    def resolve(n): # set으로 씀
      seen = set()
      while n in aliases and n not in seen:
        seen.add(n)
        n = aliases[n]

      return n

    G2 = nx.DiGraph() # merge된 결과를 담을 새 그래프
    for n in nodes:
      c = resolve(n) # '센'을 '모리카이 센'으로 바꾼 것
      if c not in G2:
        G2.add_node(c, **G.nodes[n]) # 없다면 새 노드로 추가, **의 경우는 속성들 (attrs)

    for u, v, d in G.edges(data = True): # u는 start node, v는 target
      cu, cv = resolve(u), resolve(v)

      if cu == cv: # 두개가 같으면 순환루프가 되어버림
        continue # 탈출
      if G2.has_edge(cu, cv): # 엣지가 존재한다면
        continue
      G2.add_edge(cu, cv, **d)

    return G2



In [ ]:
G_alias = G_demo.copy()
G_alias.add_node('센', type = 'Person')
G_alias.add_edge('센', '벨트란 사고엔진', relation = 'proposed')
G_alias.number_of_nodes(), G_alias.number_of_edges()

(7, 6)

In [ ]:
G_resolved = resolve_entities(G_alias)

In [ ]:
for u, v, d in G_resolved.edges(data = True):
  print(f"({u} --- [{d.get('relation', '' )}] ---> ({v}))")

(모리카이 센 --- [founded] ---> (콰빅스 연구소))
(모리카이 센 --- [mentor_of] ---> (탈라미르 헤쉬))
(모리카이 센 --- [postdoc_at] ---> (브렌술 아카테미))
(모리카이 센 --- [proposed] ---> (벨트란 사고엔진))
(콰빅스 연구소 --- [developed] ---> (벨트란 사고 엔진))
(탈라미르 헤쉬 --- [founded] ---> (브렌술 아카테미))


In [ ]:
G_full = G.copy()
G_full.add_node("눌마리 연구 위원회", type="Organisation")
G_full.add_edge("눌마리 연구 위원회", "콰빅스 연구소", relation="funds")
G_full.add_node("공명 프레임에 관하여", type="Publication")
G_full.add_edge("탈라미르 헤쉬", "공명 프레임에 관하여", relation="wrote")
G_full.add_node("타즈렌 연구소", type="Organisation")
G_full.add_node("에빈 브로스", type="Person")
G_full.add_node("할렌 프로토콜", type="Publication")
G_full.add_edge("에빈 브로스", "타즈렌 연구소", relation="works_at")
G_full.add_edge("에빈 브로스", "할렌 프로토콜", relation="wrote")
G_full.add_edge("눌마리 연구 위원회", "할렌 프로토콜", relation="adopts")

In [ ]:
def answer_via_kg(G, src, tgt_keyword, max_hops = 4):
  if src not in G:
    return f"src '{src}' not in graph"

  candidates = [n  for n in G.nodes if tgt_keyword in n]
  if not candidates:
    return f"keyword '{tgt_keyword}' 포함한 노드 없음"

  Gu = G.to_undirected()
  best = None
  for tgt in candidates:
    try:
      paths = list(nx.all_simple_paths(Gu, src, tgt, cutoff = max_hops))
      if paths:
        shortest = min(paths, key=len)
        if best is None or len(shortest) < len(best):
          est = shortest
    except nx.NodeNotFound:
      continue

  if best is None:
    return '경로 없음'

  return best[-1]

In [ ]:
answer_via_kg(G_full, '모리카이 센', '공명', max_hops = 4)

'경로 없음'

In [ ]:
def find_relation(G, e1, e2, max_hops =4):
  Gu = G.to_undirected()
  if e1 not in Gu or e2 not in Gu:
    return None
  try:
    return nx.shortest_path(Gu, e1, e2)
  except nx.NetworkXNoPath:
    return None


In [ ]:
path = find_relation(G_full, '에빈 브로스', '콰빅스 연구소')

In [ ]:
path

['에빈 브로스', '할렌 프로토콜', '눌마리 연구 위원회', '콰빅스 연구소']

In [ ]:
print(f"({len(path)-1} hop): {' -> '.join(path)}")

(3 hop): 에빈 브로스 -> 할렌 프로토콜 -> 눌마리 연구 위원회 -> 콰빅스 연구소


In [ ]:
def explain_relation(G, e1, e2, max_hops=4)-> str:
   # 에빈 브로스 -> 할렌 프로토콜 -> 눌마리 연구 위원회 -> 콰빅스 연구소 등의 관계를 같이 출력
   # 단지 화살표가 아니라 [works_at]-> 등등으로

  #  for u, v, d in G.edges():
  #   G[u][v]['relation']

In [ ]:
def explain_relation(G, e1, e2, max_hops=4) -> str:
  Gu = G.to_undirected()
  if e1 not in Gu or e2 not in Gu:
    missing = e1 if e1 not in Gu else e2
    return f" '{missing}'을 그래프에서 못찾음"
  try:
    path = nx.shortest_path(Gu, e1, e2)
  except nx.NetworkXNoPath:
    return f"'{e1}'과 '{e2}' 사이에 경로가 없음 "

  if len(path) > max_hops + 1:
    return f"경로가 너무 김 ({len(path) - 1} hops)"

  parts = [path[0]]
  for i in range(len(path) - 1):
    u, v = path[i], path[i+1]
    rel = G[u][v].get('relation') if G.has_edge(u, v) else G[v][u].get('relation', '?')
    arrow = "->" if G.has_edge(u, v) else "<-"
    parts.append(f"{arrow}[{rel}]{arrow}") # -> [works_at] ->
    parts.append(v)

  return " ".join(parts)

In [ ]:
path = explain_relation(G_full, '에빈 브로스', '콰빅스 연구소')

In [ ]:
path

'에빈 브로스 ->[wrote]-> 할렌 프로토콜 <-[adopts]<- 눌마리 연구 위원회 ->[funds]-> 콰빅스 연구소'

In [ ]:
def neighbors_to_text(G, node):
  lines = []
  for nb in G.successors(node):
    lines.append(f"({node})--[{G[node][nb].get('relation', '')}]--> ({nb})")
  for pr in G.predecessors(node):
    lines.append(f"({pr})--[{G[pr][node].get('relation', '')}]--> ({node})")

In [ ]:
print(neighbors_to_text(G_full, '모리카이 센'))

None


In [ ]:
def kg_to_text(G, focus_nodes, question):
  lines = []
  seen = set()
  for node in focus_nodes:
    if node not in G:
      continue
    for nb in G.successors(node):
